## 08 — Coverage audit & pipeline re-run

Scans `data/01_raw/` and `outputs/metrics/` to identify cities where candidate
input parquets exist but vector validation outputs are missing or incomplete.
Re-runs the pipeline for those cities, then regenerates `vector_all_cities_merged.xlsx`.

**Steps**
1. Audit: build a city × dataset coverage table (inputs vs outputs)
2. Flag cities: inputs present, sentinel missing
3. Re-run pipeline (`overwrite=False` — already-complete cities skipped automatically)
4. Verify: sentinel + tiles GPKG present for newly completed cities
5. Regenerate `vector_all_cities_merged.xlsx` from all per-city summaries

In [ ]:
!pip install -q openpyxl

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, tempfile
from pathlib import Path
import pandas as pd
import yaml

CONFIG_PATH  = Path('/content/drive/MyDrive/WorldBank/FY26 - DEP/Gates Foundation/Building Dataset Validation/configs/validation_configs.yaml')
PROJECT_ROOT = CONFIG_PATH.parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)
cfg['root_dir'] = str(PROJECT_ROOT)

DATA_DIR     = PROJECT_ROOT / cfg['data_dir']          # data/01_raw/
METRICS_ROOT = PROJECT_ROOT / 'outputs' / 'metrics'
GLOBAL_OUT   = PROJECT_ROOT / 'outputs' / 'global_metrics'
GLOBAL_OUT.mkdir(parents=True, exist_ok=True)

SENTINEL     = 'vector_metrics_tiles_all_datasets.parquet'
CITY_SUMMARY = 'vector_city_summary_all_datasets.parquet'
CAND_NAMES   = [
    d['name']
    for d in cfg.get('vector', {}).get('datasets', [])
    if d.get('enabled', True)
]

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'DATA_DIR     : {DATA_DIR}  exists={DATA_DIR.exists()}')
print(f'METRICS_ROOT : {METRICS_ROOT}  exists={METRICS_ROOT.exists()}')
print(f'Candidate datasets : {CAND_NAMES}')

In [ ]:
# ── Audit scan ───────────────────────────────────────────────────────────────
# Universe = all city slugs seen in either DATA_DIR or METRICS_ROOT.

data_cities = set()
if DATA_DIR.exists():
    for d in DATA_DIR.iterdir():
        if d.is_dir() and (d / 'vector').is_dir():
            data_cities.add(d.name.lower())

metrics_cities = set()
if METRICS_ROOT.exists():
    for d in METRICS_ROOT.iterdir():
        if d.is_dir() and any(d.iterdir()):
            metrics_cities.add(d.name.lower())

all_cities = sorted(data_cities | metrics_cities)
print(f'Cities found in DATA_DIR   : {len(data_cities)}')
print(f'Cities found in METRICS_ROOT: {len(metrics_cities)}')
print(f'Total unique cities        : {len(all_cities)}')

rows = []
for city in all_cities:
    cu = city.replace('-', '_')          # underscored form used in filenames
    data_dir_c   = DATA_DIR     / city
    metrics_dir_c = METRICS_ROOT / city
    vec_dir      = data_dir_c / 'vector'
    aoi_dir      = data_dir_c / 'aoi'
    tiles_path   = data_dir_c / 'tiles' / f'{city}_tiles.gpkg'

    row = {'city': city}

    # ── Candidate input parquets ──
    for ds in CAND_NAMES:
        files = list(vec_dir.glob(f'{cu}_{ds}*.parquet')) if vec_dir.exists() else []
        row[f'{ds}_input'] = bool(files)

    # ── Per-dataset output tile parquets ──
    for ds in CAND_NAMES:
        row[f'{ds}_output'] = (metrics_dir_c / f'vector_metrics_tiles_{ds}.parquet').exists()

    # ── Sentinel & supporting outputs ──
    row['sentinel']     = (metrics_dir_c / SENTINEL).exists()
    row['city_summary'] = (metrics_dir_c / CITY_SUMMARY).exists()
    row['tiles_gpkg']   = tiles_path.exists()

    # ── AOI file present (required for re-run) ──
    aoi_files = list(aoi_dir.glob('*.*')) if aoi_dir.exists() else []
    row['has_aoi'] = bool(aoi_files)

    # ── Reference file present ──
    cand_starts = {f'{cu}_{n}' for n in CAND_NAMES}
    ref_files = [
        f for ext in ('*.geojson', '*.gpkg', '*.shp', '*.parquet')
        for f in (vec_dir.glob(ext) if vec_dir.exists() else [])
        if not any(f.stem.lower().startswith(p) for p in cand_starts)
    ]
    row['has_ref'] = bool(ref_files)

    rows.append(row)

audit = pd.DataFrame(rows)
print(f'\nAudit complete: {len(audit)} cities')

In [ ]:
# ── Summary table & flagging ─────────────────────────────────────────────────

has_any_input  = audit[[f'{n}_input'  for n in CAND_NAMES]].any(axis=1)
has_any_output = audit[[f'{n}_output' for n in CAND_NAMES]].any(axis=1)

# Flag: candidate inputs exist but no sentinel written yet
flagged_mask = has_any_input & ~audit['sentinel']
audit['flag'] = flagged_mask

# Sub-categories
can_rerun  = audit['flag'] & audit['has_aoi'] & audit['has_ref']
no_aoi     = audit['flag'] & ~audit['has_aoi']
no_ref     = audit['flag'] & audit['has_aoi'] & ~audit['has_ref']

print('=== Coverage summary ===')
print(f'  Cities with sentinel (complete)         : {audit["sentinel"].sum()}')
print(f'  Cities with inputs but no sentinel      : {flagged_mask.sum()}')
print(f'    → can re-run (has AOI + ref)          : {can_rerun.sum()}')
print(f'    → missing AOI (cannot re-run)         : {no_aoi.sum()}')
print(f'    → missing ref file (cannot re-run)    : {no_ref.sum()}')
print(f'  Cities in METRICS_ROOT only (no inputs) : {(~has_any_input & audit["sentinel"]).sum()}')
print()

# Display the flagged cities
flagged = audit[audit['flag']].copy()
if flagged.empty:
    print('No cities flagged — all inputs have corresponding outputs.')
else:
    print(f'Flagged cities ({len(flagged)}):')
    display_cols = ['city'] + [f'{n}_input' for n in CAND_NAMES] + ['has_aoi', 'has_ref', 'sentinel', 'tiles_gpkg']
    print(flagged[display_cols].to_string(index=False))

print()
print('Full audit table saved to outputs/scratch/coverage_audit.csv')
audit.to_csv(PROJECT_ROOT / 'outputs' / 'scratch' / 'coverage_audit.csv', index=False)

In [ ]:
# ── Re-run pipeline for flagged cities ───────────────────────────────────────
# UrbanValidator.validate_vector() uses overwrite=False internally:
#   - cities with a sentinel are skipped automatically
#   - only cities loaded by load_validation_datasets() (requires AOI on disk) are processed
#
# Cities flagged but lacking AOI will not appear in validator.datasets and
# are reported separately below.

from src.validator import UrbanValidator

rerun_cities = set(audit.loc[can_rerun, 'city'])

if not rerun_cities:
    print('Nothing to re-run — no cities with inputs + AOI + ref but missing sentinel.')
else:
    print(f'Attempting re-run for {len(rerun_cities)} cities: {sorted(rerun_cities)}')
    print('(Cities already complete will be skipped automatically.)')
    print()

    # Write patched config to a temp file (adds root_dir the runner expects)
    _tmp = tempfile.NamedTemporaryFile(suffix='.yaml', delete=False, mode='w')
    yaml.dump(cfg, _tmp)
    _tmp.close()

    v = UrbanValidator(_tmp.name)

    # Filter to only the flagged cities that load_validation_datasets found
    loaded_ids = {ds['id'].lower() for ds in v.datasets}
    to_run = [ds for ds in v.datasets if ds['id'].lower() in rerun_cities]

    not_loaded = rerun_cities - loaded_ids
    if not_loaded:
        print(f'[WARN] {len(not_loaded)} flagged cities not in validator datasets'
              f' (AOI likely missing from disk): {sorted(not_loaded)}')

    print(f'Running {len(to_run)} cities through validate_vector()...')
    results = {}
    for ds in to_run:
        try:
            results[ds['id']] = v._vector_runner.run(ds)
        except Exception as e:
            print(f'[ERR] {ds["id"]}: {e}')
            results[ds['id']] = False

    print()
    print('=== Re-run results ===')
    for city_id, ok in sorted(results.items()):
        status = '[OK]  ' if ok else '[FAIL]'
        print(f'  {status} {city_id}')

    import os; os.unlink(_tmp.name)

In [ ]:
# ── Post-run verification ─────────────────────────────────────────────────────
# Re-check sentinel and tiles GPKG for all previously-flagged cities.

print('=== Post-run verification for flagged cities ===')
print(f'{"city":<35} {"sentinel":>10} {"tiles_gpkg":>12}')
print('-' * 60)
newly_complete = 0
still_missing  = []

for city in sorted(audit.loc[audit['flag'], 'city']):
    sentinel_ok  = (METRICS_ROOT / city / SENTINEL).exists()
    tiles_ok     = (DATA_DIR / city / 'tiles' / f'{city}_tiles.gpkg').exists()
    mark_s = '✓' if sentinel_ok else '✗'
    mark_t = '✓' if tiles_ok   else '✗'
    print(f'  {city:<33} {mark_s:>10} {mark_t:>12}')
    if sentinel_ok:
        newly_complete += 1
    else:
        still_missing.append(city)

print()
print(f'Newly complete  : {newly_complete}')
print(f'Still missing   : {len(still_missing)}')
if still_missing:
    print(f'  → {still_missing}')
    print('  Likely cause: AOI file was cleaned from Drive. Re-run requires re-downloading the AOI.')

In [ ]:
# ── Regenerate vector_all_cities_merged.xlsx ──────────────────────────────────
# Concatenates vector_city_summary_all_datasets.parquet from every processed city.
# Also writes .parquet and .csv for downstream use in notebook 04/06.

summaries = []
missing_summary = []

for city_dir in sorted(METRICS_ROOT.iterdir()):
    if not city_dir.is_dir():
        continue
    p = city_dir / CITY_SUMMARY
    if p.exists():
        try:
            summaries.append(pd.read_parquet(p))
        except Exception as e:
            print(f'[WARN] {city_dir.name}: could not read summary — {e}')
    else:
        missing_summary.append(city_dir.name)

if not summaries:
    print('No city summaries found — nothing to merge.')
else:
    df = pd.concat(summaries, ignore_index=True)

    stem = 'vector_all_cities_merged'
    df.to_parquet(GLOBAL_OUT / f'{stem}.parquet', index=False)
    df.to_csv(    GLOBAL_OUT / f'{stem}.csv',     index=False)

    xlsx_path = GLOBAL_OUT / f'{stem}.xlsx'
    with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name=stem, index=False)

    print(f'Merged {len(summaries)} city summaries → {len(df)} rows, {df["city"].nunique()} cities')
    print(f'  → {xlsx_path}')
    print(f'  → {GLOBAL_OUT / stem}.parquet')
    print(f'  → {GLOBAL_OUT / stem}.csv')

    if missing_summary:
        print(f'\n[WARN] {len(missing_summary)} cities in METRICS_ROOT have no city_summary parquet:')
        print(f'  {missing_summary}')

    print()
    print('=== Column list ===')
    print(df.columns.tolist())
    print()
    print('=== Cities included ===')
    print(sorted(df['city'].unique()))